In [1]:
from train import *

In [2]:
from accelerate import Accelerator
import bitsandbytes as bnb

args = Hyperparameters(batch_size=1, gradient_accumulation_steps=32, max_length=700)

accelerator = Accelerator(gradient_accumulation_steps=args.gradient_accumulation_steps)

model = model_setup(args)

train_dl, val_dl = make_dataloaders(args)

optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay
                                )
scheduler = get_scheduler(
                            "cosine",
                            optimizer=optimizer,
                            num_warmup_steps=50,
                            num_training_steps=args.epochs * len(train_dl),
                        )

model, train_dl, val_dl, optimizer, scheduler = accelerator.prepare(model, train_dl, val_dl, optimizer, scheduler)


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [3]:
model.train()

#model.gradient_checkpointing_enable()
i = iter(train_dl)
#_= next(i)
batch = next(i)
outputs = model(**batch)

loss = outputs.loss
accelerator.backward(loss)

In [4]:
accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
scheduler.step()

In [3]:
model.train()
total_loss = 0.0
n_batches = 0

for i, batch in tqdm(enumerate(train_dl)):
    if i % 20 == 0:
        print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Max Allocated: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")
        print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
            
    with accelerator.accumulate(model):
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach())
        n_batches += 1

0it [00:00, ?it/s]

Allocated: 4987.47 MB
Max Allocated: 4987.47 MB
Reserved: 5184.00 MB


21it [00:02,  9.00it/s]

Allocated: 10196.34 MB
Max Allocated: 14520.55 MB
Reserved: 15958.00 MB


41it [00:05,  8.57it/s]

...

Allocated: 15355.87 MB
Max Allocated: 19751.31 MB
Reserved: 22134.00 MB


101it [00:12,  8.72it/s]

Allocated: 15218.56 MB
Max Allocated: 19751.31 MB
Reserved: 22134.00 MB


118it [00:14,  8.34it/s]


KeyboardInterrupt: 

In [3]:
train_one_epoch(model, train_dl, optimizer, scheduler, accelerator)

  2%|▏         | 2525/116722 [04:47<3:36:24,  8.79it/s]


KeyboardInterrupt: 